# HM4SR: Review + Run Notebook

This notebook ports the main training pipeline from `run_hm4sr.py` to Jupyter.

Goals:
- Quick project sanity checks
- Optional multimodal data preprocessing
- Train and evaluate the `HM4SR` model


In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from logging import getLogger
from pathlib import Path

PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('run_hm4sr.py exists =', (PROJECT_ROOT / 'run_hm4sr.py').exists())
print('config/data.yaml exists =', (PROJECT_ROOT / 'config' / 'data.yaml').exists())
print('config/Games.yaml exists =', (PROJECT_ROOT / 'config' / 'Games.yaml').exists())


## 1) Train/eval functions (ported from `run_hm4sr.py`)


In [ ]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.data.transform import construct_transform
from recbole.utils import init_logger, get_trainer, init_seed, set_color, get_flops

def get_model(model_name: str):
    module_path = '.'.join(['recbole_model', model_name])
    model_module = importlib.import_module(module_path, __name__)
    model_class = getattr(model_module, model_name)
    return model_class

def print_result(test_result, logger, k: int = 4):
    count = 0
    info = '\ntest result:'
    for metric_name in test_result.keys():
        if count == 0:
            info += '\n'
        count = (count + 1) % k
        info += '{:15}:{:<10}    '.format(metric_name, test_result[metric_name])
    logger.info(info)

def run_recbole(model: str, dataset: str, config_file_list: list[str], saved: bool = True):
    model_cls = get_model(model)
    config = Config(model=model_cls, dataset=dataset, config_file_list=config_file_list)

    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset_obj = create_dataset(config)
    logger.info(dataset_obj)

    train_data, valid_data, test_data = data_preparation(config, dataset_obj)

    init_seed(config['seed'] + config['local_rank'], config['reproducibility'])
    model_inst = model_cls(config, train_data._dataset).to(config['device'])
    logger.info(model_inst)

    transform = construct_transform(config)
    flops = get_flops(model_inst, dataset_obj, config['device'], logger, transform)
    logger.info(set_color('FLOPs', 'blue') + f': {flops}')

    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model_inst)

    best_valid_score, best_valid_result = trainer.fit(
        train_data, valid_data, saved=saved, show_progress=config['show_progress']
    )

    test_result = trainer.evaluate(
        test_data, load_best_model=saved, show_progress=config['show_progress']
    )

    logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
    logger.info(set_color('test result', 'yellow') + f': {test_result}')
    print_result(test_result, logger, k=4)

    return {
        'best_valid_score': best_valid_score,
        'valid_metric_bigger': config['valid_metric_bigger'],
        'best_valid_result': best_valid_result,
        'test_result': test_result,
    }


## 2) Optional data preprocessing

Run this only when `dataset/Games/` is missing required files such as `txt_emb.pt`, `img_emb.pt`, or `cat.pt`.


In [ ]:
# OPTIONAL: uncomment to run preprocess
# import sys
# sys.path.insert(0, str(PROJECT_ROOT / 'dataprocess'))
#
# from args import getArgs
# from data_process import (
#     prepare_inter,
#     prepare_seq,
#     prepare_txt_emb,
#     prepare_img_emb,
#     prepare_category,
# )
#
# args = getArgs()
# prepare_inter(args)
# prepare_seq(args)
# prepare_txt_emb(args)
# prepare_img_emb(args)
# prepare_category(args)
# print('Preprocess done.')


## 2b) Inlined dataprocess source (merged into notebook)\n
\n
The next cell embeds the current `dataprocess` code so this notebook is more self-contained.\n
You can run it to register all preprocessing functions in memory.\n

In [ ]:
# Inlined dataprocess modules for single-notebook workflow
# Source: dataprocess/get_df.py
import gzip
import pandas as pd
import json

def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def get_df(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

def parse_2018(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def get_df_2018(path):
  i = 0
  df = {}
  for d in parse_2018(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

# Source: dataprocess/args.py
import argparse

def getArgs():
    parser = argparse.ArgumentParser()
    ### 基本参数
    parser.add_argument('--dataset', type=str, default='Games')
    parser.add_argument('--batch_size', type=int, default=1024)
    parser.add_argument('--max_length', type=int, default=50)
    parser.add_argument('--max_epoch', type=int, default=1000)
    parser.add_argument('--random_seed', type=int, default=2024)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument('--tolerance', type=int, default=10)
    ### 模型参数
    parser.add_argument('--hidden_dim', type=int, default=64)
    parser.add_argument('--seq_head', type=int, default=2)
    parser.add_argument('--seq_layers', type=int, default=2)
    parser.add_argument('--dropout', type=float, default=0.5)

    args = parser.parse_args()
    ### 路径参数
    args.txt_emb = f'./dataset/{args.dataset}/txt_emb.pt'
    args.img_emb = f'./dataset/{args.dataset}/img_emb.pt'
    args.ckpt = f'./ckpt/{args.dataset}'
    args.item_dict_path = f'./dataset/{args.dataset}/item2id'
    args.log_path = f'./log/{args.dataset}_{args.random_seed}.txt'
    args.inter_path = f'./dataset/{args.dataset}/{args.dataset}.inter'
    args.data_path = f'./dataset/{args.dataset}/00_seq'
    args.txt_path = f'./dataset/{args.dataset}/content.txt'
    args.img_path = f'./dataset/{args.dataset}/image/'
    args.meta_path = f'./dataset/meta_{args.dataset}.json.gz'
    return args

# Source: dataprocess/txt_extractor.py
import torch
import html
import re
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from pytorch_transformers import BertModel, BertConfig, BertTokenizer
from get_df import get_df

def clean_text(raw_text):
    # from UniSRec
    if isinstance(raw_text, list):
        cleaned_text = ' '.join(raw_text[0])
    elif isinstance(raw_text, dict):
        cleaned_text = str(raw_text)
    else:
        cleaned_text = raw_text
    cleaned_text = html.unescape(cleaned_text)
    cleaned_text = re.sub(r'["\n\r]*', '', cleaned_text)
    cleaned_text = cleaned_text.replace('\t', ' ')
    index = -1
    while -index < len(cleaned_text) and cleaned_text[index] == '.':
        index -= 1
    index += 1
    if index == 0:
        cleaned_text = cleaned_text + '.'
    else:
        cleaned_text = cleaned_text[:index] + '.'
    if len(cleaned_text) >= 2000:
        cleaned_text = ''
    return cleaned_text

def generate_text(args, items, features):
    def not_nan(nan):
        return nan == nan
    item_text_list = []
    already_items = set()
    data_df = get_df(args.meta_path)

    for index in tqdm(range(data_df.shape[0]), desc='Generate text'):
        item = data_df['asin'][index]
        if item in items and item not in already_items:
            already_items.add(item)
            text = ''
            for meta_key in features:
                if meta_key in data_df.columns:
                    content = data_df[meta_key][index]
                    if not_nan(content):
                        meta_value = clean_text(content)
                        text += meta_value + ' '
            item_text_list.append((item, text))
    with open(args.txt_path, 'w') as f:
        for i, record in enumerate(item_text_list):
            line = '\t'.join(record)
            f.write(line + '\n')
    return item_text_list

def load_content(args):
    item_text_list = []
    with open(args.txt_path, 'r') as f:
        while True:
            line = f.readline().strip()
            if len(line) == 0: break
            item, text = line.split('\t')
            item_text_list.append((item, text))
    return item_text_list

def txt_extractor(sentences, args, padding_idx=0):
    # 要求sentence必须事先按id排好序
    tokenizer = BertTokenizer.from_pretrained('./pretrained/bert-base-uncased/vocab.txt')
    config = BertConfig.from_pretrained('./pretrained/bert-base-uncased/config.json')
    bert = BertModel.from_pretrained('./pretrained/bert-base-uncased/pytorch_model.bin', config=config).to(args.device)
    result = []
    with torch.no_grad():
        for _, sentence in tqdm(enumerate(sentences), desc='Text Extracting', total=len(sentences)):
            sentence = '[CLS] ' + sentence
            token_seq = tokenizer.tokenize(sentence)
            idx_seq = tokenizer.convert_tokens_to_ids(token_seq)
            idx_seq_tensor = torch.tensor(idx_seq, dtype=torch.long).to(args.device).view(1, -1)
            output = bert(idx_seq_tensor)
            result.append(output[0][:, 0, :].cpu())
        result.insert(padding_idx, torch.zeros(result[-1].shape, dtype=torch.float))
        txt_emb = torch.cat(result, dim=0)
    torch.save(txt_emb, args.txt_emb)


# Source: dataprocess/img_extractor.py
import torch
import os
from tqdm import tqdm
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from timm.models.vision_transformer import vit_base_patch16_clip_224

class ImgDataset(Dataset):
    def __init__(self, images_path: list, images_class: list, transform=None):
        self.images_path = images_path
        self.images_class = images_class
        self.transform = transform

    def __len__(self):
        return len(self.images_path)

    def __getitem__(self, item):
        img = Image.open(self.images_path[item])
        label = self.images_class[item]
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    @staticmethod
    def collate_fn(batch):
        images, labels = tuple(zip(*batch))
        images = torch.stack(images, dim=0)
        labels = torch.as_tensor(labels)
        return images, labels

class ImgExtractTool:
    def __init__(self,
                 args,
                 model_path='./pretrained/vit_base_patch16_clip_224.pth'):
        self.device = args.device
        self.model_path = model_path
        self.basic_path = f'./dataset/{args.dataset}/image/'
        self.feature_path = f'./dataset/{args.dataset}/feature/'
        self.model = self.load_weight()
        self.data_transform = {
            'val': transforms.Compose([transforms.Resize(256),
                                       transforms.CenterCrop(224),
                                       transforms.ToTensor(),
                                       transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])}

    def load_weight(self):
        model = vit_base_patch16_clip_224()
        weights_dict = torch.load(self.model_path)
        print(model.load_state_dict(weights_dict, strict=False))
        return model.to(self.device)

    @staticmethod
    def replace_RGB(file_path):
        img = Image.open(file_path)
        if img.mode != 'RGB':
            # print("image: {} isn't RGB mode.".format(file_path))
            img_rgb = img.convert("RGB")
            os.remove(file_path)
            img_rgb.save(file_path)

    def extract_one_instance(self, instance):
        # 这里每个物品有且只有一个图片，所以也不用考虑batch的问题
        target_path = self.basic_path + instance + '/'
        image_name = os.listdir(target_path)[0]
        image_path = os.path.join(target_path, image_name)
        try:
            self.replace_RGB(image_path)
        except:
            return torch.zeros((1, 768), dtype=torch.float)

        dataset = ImgDataset([image_path], [0], transform=self.data_transform['val'])
        assert len(dataset) <= 1
        dataloader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=dataset.collate_fn)
        for _, data in enumerate(dataloader): # 实际上只会有一个batch
            img, _ = data
            output = self.model.forward_features(img.to(self.device)).cpu()
            feature = output[:, 0].view(1, 768)
            return feature

        return torch.zeros((1, 768), dtype=torch.float)

def img_extractor(args, item_id_list, padding_idx=0):
    # 要求item_id_list必须事先按id排好序
    with torch.no_grad():
        tool = ImgExtractTool(args)
        result = []
        for _, t in tqdm(enumerate(item_id_list), desc='Image Extracting', total=len(item_id_list)):
            result.append(tool.extract_one_instance(t))
        result.insert(padding_idx, torch.zeros((1, 768), dtype=torch.float))
        img_emb = torch.cat(result, dim=0)
    torch.save(img_emb, args.img_emb)



# Source: dataprocess/data_process.py
import os
import pickle
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
# inlined: generate_text, txt_extractor, load_content
# inlined: img_extractor
# inlined: get_df
from tqdm import tqdm

def amazon(args):
    data_path = f'./dataset/reviews_{args.dataset}_5.json.gz'
    data_df = get_df(data_path)
    inter_df = data_df.rename(
        columns={'reviewerID': 'user', 'asin': 'item', 'unixReviewTime': 'timestamp', 'overall': 'stars'})
    inter_df = inter_df[['user', 'item', 'timestamp']]
    user_list = sorted(inter_df['user'].unique())
    user2id = dict(zip(user_list, range(1, len(user_list) + 1)))
    inter_df['user'] = inter_df['user'].apply(lambda x: user2id[x])
    return inter_df

def inter2txt(inter_df, txt_path):
    df = inter_df.sort_values(by=['user', 'timestamp'], kind='mergesort').reset_index(drop=True)
    with open(txt_path, 'w') as f:
        f.write('user_id:token\titem_id:token\ttimestamp:float\n')
        for i, row in tqdm(df.iterrows(), desc='Generating inter file', total=df.shape[0]):
            user, item, t = row['user'], row['item'], row['timestamp']
            f.write('{}\t{}\t{}\n'.format(user, item, t))

def recbole2local(config, dataloader, local_path):
    uid_list, seq, target, interval, length = [], [], [], [], []
    user_field = config["USER_ID_FIELD"]
    seq_field = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
    target_field = config["ITEM_ID_FIELD"]
    length_field = config["ITEM_LIST_LENGTH_FIELD"]
    interval_field = config["TIME_FIELD"] + config["LIST_SUFFIX"]
    for _, interaction in enumerate(dataloader):
        uid_list.append(interaction[user_field].long())
        seq.append(interaction[seq_field].long())
        target.append(interaction[target_field].long())
        interval.append(interaction[interval_field].long())
        length.append(interaction[length_field].long())
    uid_list, seq, target, interval, length = torch.cat(uid_list, dim=0), torch.cat(seq, dim=0), torch.cat(target, dim=0), torch.cat(interval, dim=0), torch.cat(length, dim=0)
    with open(local_path, 'wb') as f: pickle.dump((uid_list, seq, target, interval, length), f)

# the dataloaders of training and testing from recbole are unexpectedly different
def recbole2local_val(config, dataloader, local_path):
    uid_list, seq, target, interval, length = [], [], [], [], []
    user_field = config["USER_ID_FIELD"]
    seq_field = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
    target_field = config["ITEM_ID_FIELD"]
    length_field = config["ITEM_LIST_LENGTH_FIELD"]
    interval_field = config["TIME_FIELD"] + config["LIST_SUFFIX"]
    for _, inter in enumerate(dataloader):
        interaction = inter[0]
        uid_list.append(interaction[user_field].long())
        seq.append(interaction[seq_field].long())
        target.append(interaction[target_field].long())
        interval.append(interaction[interval_field].long())
        length.append(interaction[length_field].long())
        length.append(interaction[length_field].long())
    uid_list, seq, target, interval, length = torch.cat(uid_list, dim=0), torch.cat(seq, dim=0), torch.cat(target, dim=0), torch.cat(interval, dim=0), torch.cat(length, dim=0)
    with open(local_path, 'wb') as f: pickle.dump((uid_list, seq, target, interval, length), f)

def prepare_inter(args):
    if not os.path.exists(args.inter_path):
        inter_df = amazon(args)
        inter2txt(inter_df, args.inter_path)

def prepare_txt_emb(args):
    if not os.path.exists(args.txt_emb):
        with open(args.item_dict_path, 'rb') as f: item2id = pickle.load(f)
        if not os.path.exists(args.txt_path):
            item_sentences = generate_text(args, item2id.keys(), ['title', 'categories', 'brand']) # omit category for future use
        else: item_sentences = load_content(args)
        item_sentences = sorted(item_sentences, key=lambda x: item2id[x[0]])
        sentences = [s[1] for s in item_sentences]
        txt_extractor(sentences, args)

def prepare_img_emb(args):
    with open(args.item_dict_path, 'rb') as f: item2id = pickle.load(f)
    item2id.pop('[PAD]')
    if not os.path.exists(args.img_emb):
        item_id_list = list(item2id.keys())
        item_id_list = sorted(item_id_list, key=lambda x: item2id[x])
        img_extractor(args, item_id_list)

def local_timestamp(args, data_path):
    with open(args.data_path.replace('00_seq', args.dataset + '.inter'), 'r') as f:
        line = f.readlines()[1:]
    tmp = [-1, 0]
    max_interval = 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        if user != tmp[0]:
            tmp = [user, timestamp]
        else:
            interval = timestamp - tmp[1]
            tmp[1] = timestamp
            max_interval = max(interval, max_interval)
    max_interval = torch.log2(torch.tensor(max_interval) + 1).item()
    with open(data_path, 'wb') as f:
        print(max_interval)
        pickle.dump(max_interval, f)

def local_minmax_day(args, data_path):
    with open(args.data_path.replace('00_seq', args.dataset + '.inter'), 'r') as f:
        line = f.readlines()[1:]
    min_date, max_date = 9999999, 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        min_date = min(int(timestamp / 86400), min_date)
        max_date = max(int(timestamp / 86400), max_date)
    with open(data_path, 'wb') as f:
        print(min_date)
        print(max_date)
        pickle.dump((min_date, max_date), f)

def prepare_seq(args):
    if not os.path.exists(args.data_path.replace('00', 'train')):
        config = Config(model='SASRec', dataset=f'./dataset/{args.dataset}/{args.dataset}', config_file_list=['./config/data.yaml'])
        dataset = create_dataset(config)
        train_data, dev_data, test_data = data_preparation(config, dataset)
        # recbole对物品进行了重新映射，因此需要将新的映射改写到原来的item2id中
        item2id = train_data.dataset.field2token_id['item_id']
        with open(args.item_dict_path, 'wb') as f: pickle.dump(item2id, f)
        train_data.shuffle = False
        recbole2local(config, train_data, args.data_path.replace('00', 'train'))
        recbole2local_val(config, dev_data, args.data_path.replace('00', 'dev'))
        recbole2local_val(config, test_data, args.data_path.replace('00', 'test'))
    if not os.path.exists(args.data_path.replace('00_seq', 'interval_num')):
        local_timestamp(args, args.data_path.replace('00_seq', 'interval_num'))
    if not os.path.exists(args.data_path.replace('00_seq', 'minmax_day')):
        local_minmax_day(args, args.data_path.replace('00_seq', 'minmax_num'))

def prepare_category(args, padding_idx=0):
    if not os.path.exists(args.data_path.replace('00_seq', 'cat.pt')):
        with open(args.item_dict_path, 'rb') as f: item2id = pickle.load(f)
        data_df = get_df(args.meta_path)
        cat_dict = {}
        cat_type_dict = {}
        for i in range(data_df.shape[0]):
            item = data_df['asin'][i]
            if item in item2id.keys():
                item_id = item2id[item]
                category = data_df['categories'][i][0]
                category_id = []
                for c in category:
                    if c not in cat_type_dict.keys():
                        cat_type_dict[c] = len(cat_type_dict)
                    category_id.append(cat_type_dict[c])
                category_id = torch.tensor(category_id)
                cat_dict[item_id] = category_id
        result = []
        for i in range(1, len(item2id)):
            ht = torch.nn.functional.one_hot(cat_dict[i], num_classes=len(cat_type_dict)).sum(dim=0).view(1, -1)
            result.append(ht)
        result.insert(padding_idx, torch.zeros((1, len(cat_type_dict)), dtype=torch.long))
        result = torch.cat(result, dim=0)
        torch.save(result, args.data_path.replace('00_seq', 'cat.pt'))


## 3) Train and evaluate HM4SR


In [ ]:
DATASET = 'Games'
CONFIG_FILES = ['./config/data.yaml', f'./config/{DATASET}.yaml']

result = run_recbole(
    model='HM4SR',
    dataset=DATASET,
    config_file_list=CONFIG_FILES,
    saved=True,
)
result


## 4) Notes\n
\n
- `dataprocess` is now embedded in this notebook via an inline source cell.\n
- `recbole` is a full framework with many modules; keeping it as local package import is the practical approach.\n
- The model still depends on data and embeddings under `./dataset/<dataset>/`.\n
- If you hit GPU issues, adjust `device` in `config/data.yaml` or your CUDA setup.\n